In [ ]:
import bilby
from bilby.core.utils import random
from pprint import pprint as pp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import bilby.gw.conversion as conv
from bilby.gw.conversion import polytrope_or_causal_params_to_lambda_1_lambda_2


from gwbench import Network, injections_CBC_params_redshift, M_of_Mc_eta, f_isco_Msolar
import corner

from converse_likelihood import EOSHyperparameterLikelihood

import lalsimulation as lalsim

random.seed(123)


In [ ]:
# generate event-level injection parameters
mmin      = 1.0
mmax      = 2.0
chi_lo    = 0 # no spins
chi_hi    = 0

cosmo_dict = {'zmin':0, 'zmax':0.2, 'sampler':'uniform_comoving_volume_inversion'}

# don't really use this, since we force m1 = m2
mass_dict  = {'dist':'uniform', 'mmin':mmin, 'mmax':mmax}
# the default waveforms above are non-precessing, hence dim=1, set dim=3 for precessing waveforms like 'IMRPhenomPv2' or 'IMRPhenomPv2_NRTidalv2'
spin_dict  = {'dim':1, 'geom':'cartesian', 'chi_lo':chi_lo, 'chi_hi':chi_hi}

redshifted = 1
num_injs   = 50 # N, number of events
seed       = 29378
file_path  = None

# everything except tidal deformability stuff -> lambdas can be calculated from m1, m2 using EOS
injections_data = injections_CBC_params_redshift(cosmo_dict,mass_dict,spin_dict,redshifted,num_injs,seed,file_path)

# fixed EOS for generation

param1       = 4.183994058757201   # gamma_0
log10_p1_cgs = 35.293288373856534
param2       = 3.168076721819637   # gamma_1
log10_p2_cgs = 35.648689969687915
param3       = 1.300271383168368   # gamma_2

causal = 0


In [ ]:
# GWBench setup

np.set_printoptions(linewidth=200)

# choose 'num' or 'sym' for numerical or symbolic derivatives
derivs = 'num'

# waveform
wf_model_name = 'lal_bns'
wf_other_var_dic = {'approximant':'IMRPhenomD_NRTidalv2'}


# user's choice: with respect to which parameters to take derivatives for the Fisher analysis
# deriv_symbs_string = 'Mc eta DL tc phic iota lam_t ra dec psi'
deriv_symbs_string = 'Mc DL tc phic iota lam_t ra dec psi'

# user's choice: convert derivatives to cos or log for specific variables
conv_cos = ('dec','iota')
# conv_log = ('Mc','DL','lam_t')
conv_log = ('DL','lam_t')

# if numeric  derivatives, user's decision -> leave on
use_rot = 1

# calculate SNRs, error matrices, and errors only for the network
only_net = 1

# number of cores to use for parallelize of the calc_det_responses_derivs
# = None for no parallelization, = 2,3,4,... to allocate N cores (even numbers preferred)
num_cores = None

# options for numeric derivative calculation
if derivs == 'num':
    # user's choice: switch particular partial derivatives to be analytical, options = [DL,tc,phic,ra,dec,psi]
    # otherwise set to None
    ana_deriv_symbs_string = 'DL tc phic ra dec psi'

    # choose numdifftools parameters for numerical derivatives
    step      = 1e-6
    method    = 'central'
    order     = 2

    # only relevant for symbolic derivatives
    gen_derivs = None

# options for symbolic derivative calculation
elif derivs == 'sym':

    # user's choice: switch particular partial derivatives to be analytical, options = [DL,tc,phic,ra,dec,psi]
    # otherwise set to None
    ana_deriv_symbs_string = None

    # choose numdifftools parameters for numerical derivatives
    step      = None
    method    = None
    order     = None

    # tell the code to generate symbolic derivatives as needed (turned on this tutorial)
    # the recommendation is to precompute them externally and load them for large-scale runs
    gen_derivs = True

In [ ]:
# event-level runs

covs  = []
means = []

for inj_id in range(num_injs):

    # randomly sample m = m1 = m2, and calculate lambdas for it
    m = np.random.uniform(mmin, mmax)

    lam_1, lam_2, eos_check = polytrope_or_causal_params_to_lambda_1_lambda_2(
                param1=param1,
                param2=param2,
                param3=param3,
                log10_pressure1_cgs=log10_p1_cgs,
                log10_pressure2_cgs=log10_p2_cgs,
                mass_1_source=m,
                mass_2_source=m,
                causal=causal
            )
    
    if not eos_check:
        print("mass ", m, " failed ")
        continue

    inj_params = {
        'Mc'    : conv.component_masses_to_chirp_mass(m, m),
        'eta'   : conv.component_masses_to_symmetric_mass_ratio(m, m),
        'chi1x' : injections_data[2][inj_id],
        'chi1y' : injections_data[3][inj_id],
        'chi1z' : injections_data[4][inj_id],
        'chi2x' : injections_data[5][inj_id],
        'chi2y' : injections_data[6][inj_id],
        'chi2z' : injections_data[7][inj_id],
        'DL'    : injections_data[8][inj_id],
        'tc'    : 0.,
        'phic'  : 0.,
        'iota'  : injections_data[9][inj_id],
        'ra'    : injections_data[10][inj_id],
        'dec'   : injections_data[11][inj_id],
        'psi'   : injections_data[12][inj_id],
        'z'     : injections_data[13][inj_id],
        'lam_t' : conv.lambda_1_lambda_2_to_lambda_tilde(lam_1, lam_2, m, m),
        'delta_lam_t' : conv.lambda_1_lambda_2_to_delta_lambda_tilde(lam_1, lam_2, m, m),
        }

    # print('injections parameter: ', inj_params)
    # print()

    # network_spec = ['CE-40_C','CE-40_S']
    network_spec = 'E' # ET

    # print('network spec: ', network_spec)
    # print()

    f_lo = 1.
    f_hi = f_isco_Msolar(M_of_Mc_eta(inj_params['Mc'],inj_params['eta']))
    df   = 2.**-4
    f    = np.arange(f_lo,f_hi+df,df)

    # print('f_lo:', f_lo, '   f_hi:', f_hi, '   df:', df)
    # print()

    # initialize Network and do general setup
    net = Network(network_spec, logger_name='CSU', logger_level='INFO')
    
    # pass all the needed and optional variables
    net.set_net_vars(wf_model_name=wf_model_name, wf_other_var_dic=wf_other_var_dic, 
                    f=f, inj_params=inj_params, deriv_symbs_string=deriv_symbs_string,
                    conv_cos=conv_cos, conv_log=conv_log, use_rot=use_rot,
                    ana_deriv_symbs_string=ana_deriv_symbs_string)

    # start the actual analysis
    net.calc_errors(only_net=only_net, derivs=derivs, step=step, method=method, order=order, gen_derivs=gen_derivs, num_cores=num_cores)

    # extract only chirp mass and lam_t
    keep = ['Mc', 'log_lam_t']
    idx = [list(net.deriv_variables).index(p) for p in keep]
    cov_marginal = net.cov[np.ix_(idx, idx)]

    mean_marginal = np.array([
        net.inj_params['Mc'],
        np.log(net.inj_params['lam_t'])
    ])

    covs.append(cov_marginal)
    means.append(mean_marginal)

# covs and means for all N = num_injs events
covs  = np.array(covs)   
means = np.array(means) 


In [ ]:
# save to use in hierarchical run
np.savez('50_runs_test.npz', covs=covs, means=means)

# to recover from file
# data  = np.load('50_runs_test.npz')
# covs  = data['covs']   # shape: (N, 2, 2)
# means = data['means']  # shape: (N, 2)